# English Accent Classification Tool

## Purpose
This tool analyzes spoken English audio to predict the speaker's regional accent (e.g., American, British, Indian). It processes audio from URLs (like YouTube, Vimeo) or uploaded files and uses a pre-trained SpeechBrain model for classification. It's intended for internal use, potentially for evaluating spoken English during hiring processes.

## Prerequisites
* **Environment:** Designed to run in Google Colab.
* **Model Download:** Requires approximately 1.3 GB download for the classification model (`Jzuluaga/accent-id-commonaccent_xlsr-en-english` from Hugging Face Hub) upon first run (Cell 3). The model is cached in `/content/models_cache`.
* **Permissions:** Ensure you have the necessary permissions if using private URLs.
* **Dependencies:** Requires Python libraries like `speechbrain`, `torchaudio`, `yt-dlp`, `ffmpeg-python`, `transformers`, `omegaconf`, `soundfile`. It also requires the `ffmpeg` system package. These are installed in Cell 1.

## How to Run
Follow these steps sequentially within a Colab session:

1.  **Cell 1 (Setup):** Run this cell once per session. It installs/updates necessary Python libraries and the `ffmpeg` system package.
2.  **Cell 2 (Code Definitions):** Run this cell once per session after Cell 1. It defines configuration constants, the `AccentClassifier` class, helper functions for downloading/processing audio, and optional VAD functions.
3.  **Cell 3 (Initialize Classifier):** Run this cell once per session after Cells 1 & 2. It instantiates the `AccentClassifier` and loads the pre-trained model from the cache or downloads it (~1.3 GB) if run for the first time. **Wait for the "✅ Model is loaded and ready..." message.**
4.  **Cell 4 (Core Processing Functions):** Run this cell once per session after Cell 3. It defines functions for handling the processing workflow and displaying results with status messages.
5.  **Choose Input Method:**
    * **Cell 5 (Classify from URL):** Enter a video/audio URL (e.g., YouTube, Vimeo). Select optional preprocessing (Noise Reduction, Silero VAD). Run the cell to start the process.
    * **Cell 6 (Classify from File):** Run the cell. Use the widget to upload an audio file (e.g., `.wav`, `.mp3`, `.m4a`). Select optional preprocessing. Classification starts automatically after the upload completes.
    * **Cell 7 (Batch Classify from URLs):** Run the cell. Upload a `.txt` file containing one URL per line. Select optional preprocessing. The tool processes each URL sequentially and saves results to a CSV file, offering it for download.

## Input
* **URL (Cell 5/7):** A direct URL to a video or audio file (e.g., YouTube, Vimeo) containing English speech.
* **File Upload (Cell 6):** Common audio formats (e.g., `.wav`, `.mp3`, `.m4a`) containing English speech.
* **Batch File (Cell 7):** A `.txt` file with one URL per line.

## Preprocessing Options (Optional)
These can be enabled in Cells 5, 6, and 7 before running classification:

* **Basic Noise Reduction (`use_noise_reduction` / `upload_use_noise_reduction`):**
    * **What it does:** Uses `torchaudio.functional.vad` to attempt removing silence or very low-level noise based on a decibel threshold (`TORCHAUDIO_VAD_TRIGGER_LEVEL`). It essentially keeps segments above the threshold.
    * **When to use:** Useful for audio with distinct periods of silence (especially leading/trailing) or low background noise that you want to remove. It might be less effective if speech and noise levels are similar or overlap significantly. It might be skipped if your `torchaudio` version is too old or if it results in empty audio.

* **Silero VAD (`use_vad` / `upload_use_vad`):**
    * **What it does:** Uses the dedicated Silero VAD model (`snakers4/silero-vad`) to identify and extract only the segments predicted to contain actual speech, discarding silence and non-speech noise based on a confidence threshold (`SILERO_VAD_THRESHOLD`).
    * **When to use:** Generally more robust for isolating speech in noisy environments or audio containing significant non-speech sounds (music, background chatter, effects). Use this if you want to focus the classification *only* on the spoken parts, potentially improving accuracy if the non-speech parts are long or loud. It might discard speech if the threshold is too high or the speech quality is very poor. It will be skipped if the VAD model detects no speech at all.

**Recommendation:** Start without VAD. If results seem poor due to silence or background noise, try "Basic Noise Reduction" first. If noise is still problematic, try "Silero VAD". Using both might be redundant or overly aggressive, potentially removing useful speech segments.

## Output
The tool outputs the following in the Colab cell:
* **Predicted Accent:** The most likely English accent detected (e.g., "American English").
* **Confidence:** The model's confidence score (0-100%) for the prediction.
* **Explanation:** A brief text describing the result, confidence level, and potentially mentioning the next most likely accent or typical features.
* **Top Predictions:** A ranked list of the top 5 most likely accents and their confidence scores.
* For batch processing (Cell 7), results are also saved to a timestamped CSV file.

## Troubleshooting
* **Model Download Errors (Cell 3):** Check internet connection. Ensure Hugging Face Hub is accessible. Re-run Cell 3. Ensure `MODEL_CACHE_DIR` is writable.
* **`ffmpeg` / `yt-dlp` Errors:** Usually related to installation (Cell 1) or invalid URLs/formats. Restart runtime and re-run Cell 1 if installation errors persist. Check URL validity.
* **Long Runtimes:** Model download (first time), processing long audio/video, or batch processing takes time.
* **Low Confidence / Incorrect Prediction:** Can be due to poor audio quality, unclear speech, heavy background noise, short speech duration, or accents underrepresented in the model's training data. Try applying VAD options.
* **FFmpeg/ffprobe Errors during Processing:** Might indicate corrupted audio files or unsupported codecs not handled by `yt-dlp` or `ffmpeg`.
* **VAD Errors:** Silero VAD requires specific Pytorch versions and can sometimes fail; basic VAD depends on `torchaudio` version. If one fails, try running without it or using the other.

## Model Information
* **Source:** `Jzuluaga/accent-id-commonaccent_xlsr-en-english` on Hugging Face Hub.
* **Cache Directory:** `/content/models_cache`.
* **Interface:** Uses a custom SpeechBrain interface (`custom_interface.py`, `CustomEncoderWav2vec2Classifier`).

In [ ]:
#@title 1. Setup: Install Dependencies & Import Libraries
#@markdown Run this cell once per session to install required packages.

# Install necessary Python packages quietly
!pip install speechbrain==0.5.16 torchaudio yt-dlp ffmpeg-python transformers omegaconf soundfile --quiet

# Install ffmpeg system package (needed for audio processing) quietly
!apt-get update -qq && apt-get install -y ffmpeg -qq > /dev/null

print("✅ Dependencies installed/updated.")

# Import necessary libraries
import os
import sys
import logging
import torch
import torchaudio
import subprocess
import tempfile
import shutil # For temporary directory management
import contextlib # For context manager for temp dir
import uuid
import yt_dlp
from speechbrain.pretrained.interfaces import foreign_class
from google.colab import files # For file upload feature
import IPython.display as ipd # To play audio if needed
from omegaconf import OmegaConf # For Silero VAD utils
import soundfile as sf
import numpy as np

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.getLogger('speechbrain').setLevel(logging.WARNING) # Reduce SpeechBrain verbosity
logging.getLogger('yt_dlp').setLevel(logging.WARNING) # Reduce yt-dlp verbosity
logger = logging.getLogger("AccentClassifierTool")

# Define cache directory for models
MODEL_CACHE_DIR = "/content/models_cache" # Store models in Colab's environment
os.makedirs(MODEL_CACHE_DIR, exist_ok=True)
logger.info(f"Using model cache directory: {MODEL_CACHE_DIR}")

# --- Temporary Directory Context Manager ---
@contextlib.contextmanager
def temporary_directory(*args, **kwargs):
    """Context manager for creating and cleaning up a temporary directory."""
    d = tempfile.mkdtemp(*args, **kwargs)
    try:
        yield d
    finally:
        try:
            shutil.rmtree(d)
            logger.debug(f"Cleaned up temporary directory: {d}")
        except OSError as e:
            logger.warning(f"Could not remove temporary directory {d}: {e}")

print("✅ Libraries imported and logging configured.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.3/173.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.6/630.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 211.5/211.5 MB 131.9 MB/s eta 0:00:01

In [ ]:
#@title 2. Code Definitions: Configuration, Classifier, and Helpers
#@markdown Defines configuration constants, the AccentClassifier class, and helper functions for audio processing. Run once per session.

# --- Configuration ---
CONFIG = {
    # Audio Processing
    "TARGET_SAMPLE_RATE": 16000,
    "OPTIMIZE_MAX_DURATION_S": 15, # Max duration (in seconds) for optimized audio snippet (centered trim)

    # Model & Prediction
    "MODEL_REPO": "Jzuluaga/accent-id-commonaccent_xlsr-en-english",
    "MODEL_CLASS_MODULE": "custom_interface.py",
    "MODEL_CLASS_NAME": "CustomEncoderWav2vec2Classifier",
    "NUM_TOP_PREDICTIONS": 5, # How many top accents to report in results

    # yt-dlp Download Options
    "YTDLP_SOCKET_TIMEOUT_S": 30,
    "YTDLP_FALLBACK_TIMEOUT_S": 60,

    # Optional Preprocessing (Default Parameters)
    "SILERO_VAD_THRESHOLD": 0.5, # 0.0 to 1.0, lower = more sensitive to speech
    "TORCHAUDIO_VAD_TRIGGER_LEVEL": 7.0 # dB level for torchaudio VAD trigger
}

# Define a mapping from model labels to user-friendly names
ACCENT_MAPPING = {
    "us": "American English",
    "england": "British English",
    "australia": "Australian English",
    "canada": "Canadian English",
    "scotland": "Scottish English",
    "ireland": "Irish English",
    "indian": "Indian English",
    "newzealand": "New Zealand English",
    "southatlandtic": "South Atlantic English", # Note: Often refers to specific UK-related accents like St Helena/Falklands
    "african": "African English", # Note: Very broad category
    "wales": "Welsh English",
    "bermuda": "Bermudian English",
    "malaysia": "Malaysian English",
    "singapore": "Singaporean English",
    "hongkong": "Hong Kong English",
    "philippines": "Philippine English"
}

# --- Accent Classifier Class ---
class AccentClassifier:
    """Class to handle accent classification using SpeechBrain."""

    def __init__(self, config, cache_dir=MODEL_CACHE_DIR):
        """Initialize the accent classifier."""
        self.config = config
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
        self.classifier = None
        self.model_loaded = False
        logger.info(f"AccentClassifier initialized. Model will be downloaded to {cache_dir} on first use.")

    def _load_model(self):
        """Load the SpeechBrain model if not already loaded."""
        if self.classifier is None:
            logger.info(f"Loading SpeechBrain model ({self.config['MODEL_REPO']})...")
            try:
                self.classifier = foreign_class(
                    source=self.config['MODEL_REPO'],
                    pymodule_file=self.config['MODEL_CLASS_MODULE'],
                    classname=self.config['MODEL_CLASS_NAME'],
                    savedir=self.cache_dir
                )
                # Perform a dummy check to ensure model components are loaded
                _ = self.classifier.hparams.label_encoder
                self.model_loaded = True
                logger.info("✅ SpeechBrain model loaded successfully!")
            except Exception as e:
                logger.error(f"❌ Failed to load SpeechBrain model: {e}", exc_info=True)
                self.model_loaded = False
                # Raise a more informative error for the user
                raise RuntimeError(f"Failed to load SpeechBrain model from {self.config['MODEL_REPO']}. Check logs for details (Network? Cache dir permissions? Model repo changes?). Original error: {e}")

    def _get_audio_info(self, audio_path):
        """Uses ffprobe to get sample rate, channels, and duration."""
        if not os.path.exists(audio_path):
             logger.error(f"Audio info check failed: File not found at {audio_path}")
             return None
        try:
            probe_cmd = [
                "ffprobe", "-v", "error",
                "-show_entries", "stream=sample_rate,channels:format=duration",
                "-of", "default=noprint_wrappers=1:nokey=1", audio_path
            ]
            output = subprocess.check_output(probe_cmd, timeout=10).decode().strip().split('\\n') # Might need \\r\\n on Windows
            # Expected output order might vary, check carefully or use JSON output
            if len(output) >= 3: # Need at least sample_rate, channels, duration
                 # Order seems to be SampleRate, Channels, Duration based on common ffprobe output
                 return {
                     "sample_rate": int(output[0]),
                     "channels": int(output[1]),
                     "duration": float(output[2])
                 }
            elif len(output) == 1 and ':' in output[0]: # Sometimes format=duration comes first
                 # Crude check if it looks like duration only
                 try:
                     duration = float(output[0])
                     # Could run separate probes for sample_rate/channels if needed
                     logger.warning("ffprobe only returned duration. Cannot check sample rate/channels for optimization.")
                     return {"duration": duration, "sample_rate": None, "channels": None}
                 except ValueError:
                     logger.error(f"Could not parse ffprobe output: {output}")
                     return None
            else:
                 logger.error(f"Unexpected ffprobe output format: {output}")
                 return None
        except FileNotFoundError:
             logger.error("Error: ffprobe not found. Make sure ffmpeg (and ffprobe) are installed (Step 1).")
             return None
        except subprocess.TimeoutExpired:
            logger.error(f"ffprobe timed out checking file: {audio_path}")
            return None
        except Exception as e:
            logger.error(f"Error running ffprobe on {audio_path}: {e}", exc_info=True)
            return None

    def optimize_audio(self, audio_path, output_dir):
        """
        Optimizes audio: Ensures 16kHz mono WAV, trims if needed.
        Skips ffmpeg conversion if input already meets criteria.
        """
        target_sr = self.config['TARGET_SAMPLE_RATE']
        max_duration = self.config['OPTIMIZE_MAX_DURATION_S']

        logger.info(f"Optimizing audio: {os.path.basename(audio_path)} (Target: {target_sr}Hz, Mono, <= {max_duration}s)")
        audio_info = self._get_audio_info(audio_path)

        if not audio_info:
            logger.warning("Could not get audio info, proceeding with ffmpeg conversion.")
            needs_conversion = True
            duration = max_duration + 1 # Assume trimming might be needed
        else:
            duration = audio_info.get("duration", max_duration + 1)
            is_target_sr = audio_info.get("sample_rate") == target_sr
            is_mono = audio_info.get("channels") == 1
            is_within_duration = duration <= max_duration
            # Crude check for WAV (more robust check would examine format name/codec)
            is_wav = os.path.splitext(audio_path)[1].lower() == '.wav'

            if is_target_sr and is_mono and is_within_duration and is_wav:
                logger.info("✅ Audio already meets criteria. Skipping ffmpeg conversion.")
                # Optional: Copy to output_dir for consistency, or just return original path
                # shutil.copy2(audio_path, os.path.join(output_dir, os.path.basename(audio_path)))
                return audio_path # Return original path if no conversion needed
            else:
                logger.info("Audio needs conversion/trimming.")
                needs_conversion = True

        # Proceed with ffmpeg if needed
        temp_id = str(uuid.uuid4())
        optimized_path = os.path.join(output_dir, f"optimized_{os.path.splitext(os.path.basename(audio_path))[0]}_{temp_id}.wav")

        try:
            trim_options = []
            if duration > max_duration:
                start_time = max(0, (duration - max_duration) / 2)
                logger.info(f"Trimming audio to {max_duration}s (centered, start: {start_time:.2f}s)")
                trim_options = ["-ss", str(start_time), "-t", str(max_duration)]
            else:
                logger.info("Audio duration is within limit.")

            cmd = [
                "ffmpeg", "-y", "-i", audio_path
            ] + trim_options + [
                "-ar", str(target_sr),
                "-ac", "1",
                "-c:a", "pcm_s16le", # Explicit WAV format
                "-af", "dynaudnorm", # Basic normalization
                "-vn",
                "-hide_banner", "-loglevel", "warning", # Quieter ffmpeg output
                optimized_path
            ]

            logger.info(f"Running FFmpeg: {' '.join(cmd)}")
            result = subprocess.run(cmd, check=False, capture_output=True, text=True, timeout=120) # Add timeout

            if result.returncode != 0:
                logger.error(f"FFmpeg optimization failed! Error: {result.stderr}")
                if os.path.exists(optimized_path): os.remove(optimized_path)
                return None
            elif not os.path.exists(optimized_path) or os.path.getsize(optimized_path) == 0:
                 logger.error(f"FFmpeg ran, but optimized file missing or empty: {optimized_path}. Stderr: {result.stderr}")
                 return None
            else:
                file_size_mb = os.path.getsize(optimized_path) / (1024 * 1024)
                logger.info(f"✅ Audio optimized successfully: {os.path.basename(optimized_path)} ({file_size_mb:.2f} MB)")
                return optimized_path

        except FileNotFoundError:
             logger.error("Error: ffmpeg not found. Ensure it's installed (Step 1).")
             return None
        except subprocess.TimeoutExpired:
            logger.error(f"FFmpeg timed out optimizing file: {audio_path}")
            if os.path.exists(optimized_path): os.remove(optimized_path)
            return None
        except Exception as e:
            logger.error(f"Error during audio optimization: {e}", exc_info=True)
            if 'optimized_path' in locals() and os.path.exists(optimized_path):
                 try: os.remove(optimized_path)
                 except OSError: pass
            return None

    def predict_accent(self, audio_path, temp_dir):
        """Predict accent from an audio file, optimizing it first."""
        if not self.model_loaded or self.classifier is None:
            logger.error("Model not loaded. Cannot predict.")
            return {"error": "Model not loaded."}

        if not os.path.exists(audio_path):
             logger.error(f"Input audio file not found: {audio_path}")
             return {"error": f"Input audio file not found: {os.path.basename(audio_path)}"}

        optimized_audio_path = None
        try:
            # Optimize audio within the provided temp_dir
            logger.info(f"Optimizing input audio: {os.path.basename(audio_path)}")
            optimized_audio_path = self.optimize_audio(audio_path, temp_dir)

            if not optimized_audio_path:
                logger.error("Audio optimization failed. Cannot proceed.")
                return {"error": "Audio optimization failed."}

            # Process the *optimized* audio
            logger.info(f"Classifying optimized audio: {os.path.basename(optimized_audio_path)}")
            out_prob, score, index, text_lab = self.classifier.classify_file(optimized_audio_path)

            probs = out_prob.squeeze().cpu().detach().numpy()
            label_encoder = self.classifier.hparams.label_encoder
            sorted_indices = probs.argsort()[::-1]

            top_accents = []
            num_preds = self.config['NUM_TOP_PREDICTIONS']
            for idx in sorted_indices[:num_preds]:
                raw_label = label_encoder.decode_ndim(torch.tensor([idx]))[0]
                friendly_name = ACCENT_MAPPING.get(raw_label, f"Unknown ({raw_label})") # Handle unknown labels
                confidence = float(probs[idx]) * 100
                top_accents.append({
                    "label": friendly_name,
                    "raw_label": raw_label,
                    "confidence": round(confidence, 2)
                })

            primary_prediction = top_accents[0] if top_accents else None
            if not primary_prediction:
                 logger.error("Classification produced no results.")
                 return {"error": "Classification failed to produce results."}

            explanation = self._generate_explanation(top_accents)

            # The optimized file will be cleaned up by the temporary_directory context manager

            return {
                "accent": primary_prediction["label"],
                "confidence": primary_prediction["confidence"],
                "top_accents": top_accents,
                "explanation": explanation
            }
        except Exception as e:
            logger.error(f"Error during accent classification: {e}", exc_info=True)
            return {"error": f"An unexpected error occurred during classification: {str(e)}"}
        # No finally block needed for optimized_audio_path cleanup if using context manager

    def _generate_explanation(self, top_accents):
        """Generate a human-readable explanation for the classification result."""
        # (Explanation generation logic remains the same as in the original notebook)
        if not top_accents: return "No results available."
        primary = top_accents[0]
        secondary = top_accents[1] if len(top_accents) > 1 else None

        if primary["confidence"] > 90: confidence_level = "very high confidence"
        elif primary["confidence"] > 75: confidence_level = "high confidence"
        elif primary["confidence"] > 50: confidence_level = "moderate confidence"
        else: confidence_level = "low confidence"

        explanation = f"The model predicts the speaker has a **{primary['label']}** accent with {confidence_level} ({primary['confidence']:.1f}%)."

        if secondary:
            confidence_diff = primary["confidence"] - secondary["confidence"]
            if confidence_diff < 15:
                 explanation += f" However, the prediction for **{secondary['label']}** is also notable ({secondary['confidence']:.1f}%), suggesting potential overlap or mixed features."
            elif confidence_diff < 40:
                 explanation += f" The next likely accent detected is **{secondary['label']}** ({secondary['confidence']:.1f}%)."

        accent_features = {
            "American English": "Typically features rhoticity (pronounced 'r's) and distinct vowel sounds.",
            "British English": "Often non-rhotic (silent 'r's after vowels) with different vowel qualities (RP is one standard).",
            "Australian English": "Known for unique vowel sounds and diphthong shifts.",
            "Indian English": "May exhibit retroflex consonants, syllable-timed rhythm, and distinct intonation.",
            "Canadian English": "Shares features with American English but includes elements like Canadian Raising.",
            "Scottish English": "Distinctive for its rhoticity, unique vowel system, and vocabulary.",
            "Irish English": "Features include rhoticity, specific vowel sounds, and variations in intonation."
            # Add more descriptions if needed based on ACCENT_MAPPING
        }
        if primary["label"] in accent_features:
            explanation += f" ({accent_features[primary['label']]})"
        return explanation

# --- Helper Functions for Audio Processing ---

def download_and_extract_audio_from_url(url, output_dir, config):
    """Downloads/extracts audio from URL to 16kHz mono WAV using yt-dlp."""
    if not url:
        logger.error("No URL provided for download.")
        return None

    file_id = str(uuid.uuid4())
    output_template = os.path.join(output_dir, f"audio_{file_id}")
    final_audio_path = output_template + ".wav"
    target_sr = config['TARGET_SAMPLE_RATE']
    timeout = config['YTDLP_SOCKET_TIMEOUT_S']
    fallback_timeout = config['YTDLP_FALLBACK_TIMEOUT_S']

    logger.info(f"Attempting download/extraction for URL: {url}")
    logger.info(f"Target temporary audio path: {os.path.basename(final_audio_path)}")

    # Method 1: Direct audio extraction
    try:
        logger.info("Trying direct WAV extraction (16kHz, mono) with yt-dlp...")
        ydl_opts_audio = {
            'format': 'bestaudio/best',
            'outtmpl': output_template,
            'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav'}],
            'postprocessor_args': ['-ar', str(target_sr), '-ac', '1'],
            'quiet': True, 'verbose': False, 'noprogress': True,
            'socket_timeout': timeout,
        }
        with yt_dlp.YoutubeDL(ydl_opts_audio) as ydl:
            ydl.download([url])

        if os.path.exists(final_audio_path) and os.path.getsize(final_audio_path) > 0:
            logger.info(f"✅ Direct audio extraction successful: {os.path.basename(final_audio_path)}")
            return final_audio_path
        else:
            logger.warning(f"Direct audio extraction finished, but file not found/empty: {os.path.basename(final_audio_path)}")
    except yt_dlp.utils.DownloadError as e:
         logger.warning(f"yt-dlp direct download error (might be URL/format issue): {e}. Trying fallback.")
    except Exception as e:
        logger.warning(f"Direct audio extraction failed unexpectedly: {e}. Trying fallback.")
        if os.path.exists(final_audio_path):
           try: os.remove(final_audio_path)
           except OSError: pass

    # Method 2: Fallback - Download best, then FFmpeg manually
    video_path_template = os.path.join(output_dir, f"video_{file_id}")
    downloaded_video_path = None
    logger.info("Fallback: Downloading best format and extracting with FFmpeg...")
    try:
        ydl_opts_video = {
            'format': 'best[ext=mp4]/bestvideo[ext=mp4]+bestaudio[ext=m4a]/best',
            'outtmpl': video_path_template + '.%(ext)s',
            'quiet': True, 'verbose': False, 'noprogress': True,
            'socket_timeout': fallback_timeout,
        }
        with yt_dlp.YoutubeDL(ydl_opts_video) as ydl:
            info_dict = ydl.extract_info(url, download=True)
            downloaded_video_path = ydl.prepare_filename(info_dict)

        if not downloaded_video_path or not os.path.exists(downloaded_video_path):
             logger.error(f"Fallback failed: File not downloaded from {url}")
             return None

        logger.info(f"Extracting audio from downloaded file: {os.path.basename(downloaded_video_path)}")
        cmd = [
            "ffmpeg", "-y", "-i", downloaded_video_path,
            "-ar", str(target_sr), "-ac", "1", "-vn",
            "-c:a", "pcm_s16le",
            "-hide_banner", "-loglevel", "error",
            final_audio_path
        ]
        result = subprocess.run(cmd, check=False, capture_output=True, text=True, timeout=180) # Longer timeout for extraction

        # Cleanup the intermediate large download immediately
        if downloaded_video_path and os.path.exists(downloaded_video_path):
             try: os.remove(downloaded_video_path); logger.debug(f"Cleaned temp download: {os.path.basename(downloaded_video_path)}")
             except OSError: pass

        if result.returncode != 0:
            logger.error(f"Fallback FFmpeg extraction failed. Error: {result.stderr}")
            return None

        if os.path.exists(final_audio_path) and os.path.getsize(final_audio_path) > 0:
            logger.info(f"✅ Fallback audio extraction successful: {os.path.basename(final_audio_path)}")
            return final_audio_path
        else:
             logger.error(f"Fallback FFmpeg ran but output audio not found/empty: {os.path.basename(final_audio_path)}")
             return None
    except yt_dlp.utils.DownloadError as e:
         logger.error(f"Fallback download failed: {e}")
         return None
    except FileNotFoundError:
        logger.error("Fallback failed: yt-dlp or FFmpeg not found.")
        return None
    except subprocess.TimeoutExpired:
        logger.error("Fallback FFmpeg extraction timed out.")
        return None
    except Exception as e:
        logger.error(f"Error during fallback download/extraction: {e}", exc_info=True)
        return None
    finally:
         # Ensure intermediate file is cleaned up even if extraction fails later
        if 'downloaded_video_path' in locals() and downloaded_video_path and os.path.exists(downloaded_video_path):
            try: os.remove(downloaded_video_path)
            except OSError: pass

# --- Optional Preprocessing Functions ---

def apply_vad_silero(input_audio_path, output_dir, config):
    """Applies Silero VAD to keep only speech segments."""
    logger.info("Applying Silero Voice Activity Detection (VAD)...")
    vad_output_path = os.path.join(output_dir, f"vad_silero_{os.path.basename(input_audio_path)}")
    target_sr = config['TARGET_SAMPLE_RATE'] # VAD model expects 16k

    try:
        # Ensure PyTorch uses single thread for stability with Silero VAD
        # Check current threads and set temporarily if needed
        original_threads = torch.get_num_threads()
        torch.set_num_threads(1)

        model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                                      model='silero_vad',
                                      force_reload=False, # Avoid re-download unless needed
                                      trust_repo=True) # Explicitly trust the repo
        (get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

        # Read audio, ensuring target sample rate for the model
        try:
             # Silero's read_audio handles resampling
            wav = read_audio(input_audio_path, sampling_rate=target_sr)
            current_sr = target_sr
        except Exception as read_err:
            logger.warning(f"Silero read_audio failed ({read_err}), trying torchaudio/resample...")
            waveform, sample_rate = torchaudio.load(input_audio_path)
            if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
            if sample_rate != target_sr:
                resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=target_sr)
                waveform = resampler(waveform)
            wav = waveform.squeeze(0) # Shape should be (N,)
            current_sr = target_sr

        # Get speech timestamps
        speech_timestamps = get_speech_timestamps(
            wav, model, sampling_rate=current_sr,
            threshold=config['SILERO_VAD_THRESHOLD'] # Use configured threshold
        )

        # Restore original PyTorch thread count
        torch.set_num_threads(original_threads)

        if not speech_timestamps:
             logger.warning("⚠️ No speech detected by Silero VAD. Returning None for VAD step.")
             return None

        # Collect speech chunks and save
        vad_waveform = collect_chunks(speech_timestamps, wav)
        save_audio(vad_output_path, vad_waveform, sampling_rate=current_sr)

        # Check if output is valid
        if not os.path.exists(vad_output_path) or os.path.getsize(vad_output_path) == 0:
             logger.error("VAD processing ran but output file is missing or empty.")
             return None

        logger.info(f"✅ VAD processed audio saved: {os.path.basename(vad_output_path)}")
        return vad_output_path

    except Exception as e:
         logger.error(f"Error during Silero VAD processing: {e}", exc_info=True)
         # Ensure threads are reset even on error
         try: torch.set_num_threads(original_threads)
         except NameError: pass # If original_threads wasn't set
         return None

def apply_basic_noise_reduction(input_audio_path, output_dir, config):
    """Applies basic noise/silence reduction using torchaudio VAD."""
    logger.info("Applying basic Noise/Silence Reduction (torchaudio.functional.vad)...")
    processed_output_path = os.path.join(output_dir, f"nr_vad_{os.path.basename(input_audio_path)}")
    target_sr = config['TARGET_SAMPLE_RATE'] # torchaudio VAD works best at specific rates

    try:
        waveform, sample_rate = torchaudio.load(input_audio_path)

        # Ensure mono
        if waveform.shape[0] > 1:
             waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Resample if needed for VAD function
        if sample_rate != target_sr:
             logger.debug(f"Resampling for torchaudio VAD: {sample_rate}Hz -> {target_sr}Hz")
             resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=target_sr)
             wf_for_vad = resampler(waveform)
             current_sr = target_sr
        else:
             wf_for_vad = waveform
             current_sr = sample_rate

        # Apply torchaudio VAD - returns waveform *without* silence/noise below threshold
        processed_waveform = torchaudio.functional.vad(
            wf_for_vad, sample_rate=current_sr,
            trigger_level=config['TORCHAUDIO_VAD_TRIGGER_LEVEL'] # Use configured level
            # Other parameters like 'noise_reduction_amount' could be added if available/needed
         )

        if processed_waveform.numel() == 0:
             logger.warning("⚠️ Noise reduction (torchaudio VAD) resulted in empty audio. Skipping.")
             # Return None to indicate step should be skipped, main pipeline will use previous audio
             return None

        # Save the processed audio
        torchaudio.save(processed_output_path, processed_waveform, current_sr)
        logger.info(f"✅ Noise/Silence reduction audio saved: {os.path.basename(processed_output_path)}")
        return processed_output_path

    except AttributeError:
         # Handles cases where torchaudio version might be too old for functional.vad
         logger.warning("⚠️ Skipping noise reduction: `torchaudio.functional.vad` not available or failed. Check torchaudio version.")
         return None # Indicate failure/skip
    except Exception as e:
         logger.error(f"Error during basic noise reduction: {e}", exc_info=True)
         return None # Indicate failure


print("✅ Code definitions loaded.")

✅ Code definitions loaded.


In [ ]:
#@title 3. Initialize Classifier and Load Model
#@markdown Instantiates the classifier and loads the pre-trained model weights.
#@markdown **This cell downloads ~1.3 GB the first time it's run.**

# Instantiate the classifier using the global CONFIG
try:
    global_classifier = AccentClassifier(config=CONFIG, cache_dir=MODEL_CACHE_DIR)
    # Explicitly load the model now (triggers download if needed)
    global_classifier._load_model()
    MODEL_READY = global_classifier.model_loaded
except Exception as e:
    # Error is likely raised from _load_model now
    logger.critical(f"🚨 FATAL ERROR during model initialization/loading: {e}")
    MODEL_READY = False
    global_classifier = None # Ensure classifier object is None if failed

if not MODEL_READY:
    print("\\n" + "="*30)
    print("🚨 ERROR: Model could not be loaded. Classification will not work.")
    print("Please check the logs above for details (e.g., network issues, Hugging Face Hub access).")
    print("You may need to restart the runtime and try again.")
    print("="*30)
else:
    print("\\n✅ Model is loaded and ready for classification.")

/usr/local/lib/python3.11/dist-packages/speechbrain/utils/checkpoints.py:147: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(path, map_location=device), strict=Fal

\n✅ Model is loaded and ready for classification.


In [ ]:
#@title 4. Core Processing and Display Functions (Enhanced UX)
#@markdown Defines the main functions to handle audio processing and display results, with status messages and color. **Run this cell once.**

import time # For potential small delays if needed

# --- ANSI Color Codes ---
class Colors:
    RESET = '\033[0m'
    BOLD = '\033[1m'
    RED = '\033[91m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    BLUE = '\033[94m'
    MAGENTA = '\033[95m'
    CYAN = '\033[96m'
    WHITE = '\033[97m'

# Helper function for printing status
def print_status(message, color=Colors.CYAN):
    print(f"{color}⏳ {message}{Colors.RESET}")
    # Optional: Flush output buffer to ensure message appears immediately
    sys.stdout.flush()

# Helper function for printing section headers
def print_header(title, color=Colors.BLUE):
    print(f"\n{color}{Colors.BOLD}{'='*10} {title} {'='*10}{Colors.RESET}")

# Helper function for printing errors
def print_error(message):
     print(f"{Colors.RED}❌ Error: {message}{Colors.RESET}")

# --- Updated Display Function ---
def _display_results(result_dict):
    """Formats and prints the classification results with colors."""
    print_header("Classification Results", color=Colors.MAGENTA)

    if not result_dict:
        print_error("No results dictionary received.")
    elif "error" in result_dict:
        print_error(f"During classification: {result_dict['error']}")
    elif 'accent' not in result_dict:
        log_msg = f"Classification returned unexpected result format: {result_dict}"
        logger.error(log_msg)
        print_error(f"Classification results missing 'accent' key. Result was: {result_dict}")
    else:
        pred_accent = result_dict.get('accent', 'N/A')
        pred_conf = result_dict.get('confidence', 0.0)
        explanation = result_dict.get('explanation', 'No explanation provided.')
        top_accents = result_dict.get('top_accents', [])

        # --- Predicted Accent ---
        print(f"🗣️ {Colors.BOLD}Predicted Accent:{Colors.RESET} {Colors.GREEN}{pred_accent}{Colors.RESET}")

        # --- Confidence ---
        try:
            confidence_str = f"{float(pred_conf):.1f}%"
            confidence_color = Colors.GREEN if float(pred_conf) > 75 else (Colors.YELLOW if float(pred_conf) > 50 else Colors.RED)
            print(f"📊 {Colors.BOLD}Confidence:{Colors.RESET} {confidence_color}{confidence_str}{Colors.RESET}")
        except (ValueError, TypeError):
             print(f"📊 {Colors.BOLD}Confidence:{Colors.RESET} {Colors.RED}{pred_conf} (Invalid Format){Colors.RESET}")

        # --- Explanation ---
        print(f"\n📝 {Colors.BOLD}Explanation:{Colors.RESET}")
        print(f"{Colors.YELLOW}{explanation}{Colors.RESET}") # Yellow for informational text

        # --- Top Predictions ---
        if top_accents:
            print(f"\n🔝 {Colors.BOLD}Top {len(top_accents)} Predictions:{Colors.RESET}")
            for i, acc in enumerate(top_accents):
                 label = acc.get('label', 'Unknown Label')
                 conf = acc.get('confidence', 0.0)
                 try:
                      # Use slightly dimmer color for secondary predictions
                      print(f"  {i+1}. {Colors.WHITE}{label}{Colors.RESET} ({Colors.CYAN}{float(conf):.1f}%{Colors.RESET})")
                 except (ValueError, TypeError):
                      print(f"  {i+1}. {Colors.WHITE}{label}{Colors.RESET} ({Colors.RED}{conf} - Invalid Format{Colors.RESET})")
        else:
            print(f"\n{Colors.YELLOW}ℹ️ No detailed top predictions list available.{Colors.RESET}")
    print(f"{Colors.MAGENTA}{'='*46}{Colors.RESET}") # Footer separator


# --- Updated Processing Function ---
def _process_audio_source(source_type, source_data, temp_dir, config, apply_nr=False, apply_vad=False):
    """
    Handles audio acquisition and optional preprocessing with status updates.
    Returns the path to the final audio file ready for classification, or None on failure.
    """
    initial_audio_path = None
    current_audio_path = None

    # 1. Get Initial Audio File
    if source_type == 'url':
        url = source_data
        print_status(f"Downloading/Extracting audio from URL...")
        initial_audio_path = download_and_extract_audio_from_url(url, temp_dir, config)
        if initial_audio_path:
            logger.info(f"Audio downloaded/extracted to: {os.path.basename(initial_audio_path)}")
        else:
            print_error("Failed to get audio from URL.")
            return None
    elif source_type == 'file':
        file_name, file_content = source_data
        print_status(f"Saving uploaded file '{file_name}'...")
        # Create a unique name
        unique_id = str(uuid.uuid4()).split('-')[0]
        safe_file_name = "".join(c for c in file_name if c.isalnum() or c in ('_', '.', '-')).strip()
        initial_audio_path = os.path.join(temp_dir, f"uploaded_{unique_id}_{safe_file_name}")
        try:
            with open(initial_audio_path, 'wb') as f:
                f.write(file_content)
            logger.info(f"Uploaded file saved temporarily to: {os.path.basename(initial_audio_path)}")
        except Exception as e:
            log_msg = f"Failed to save uploaded file: {e}"
            logger.error(log_msg, exc_info=True)
            print_error(log_msg)
            return None
    else:
        log_msg = f"Invalid source_type: {source_type}"
        logger.error(log_msg)
        print_error(log_msg)
        return None

    current_audio_path = initial_audio_path
    print(f"{Colors.GREEN}✅ Initial audio ready: {os.path.basename(current_audio_path)}{Colors.RESET}")

    # 2. Apply Basic Noise Reduction (Optional)
    if apply_nr:
        print_status("Applying Noise Reduction...")
        nr_path = apply_basic_noise_reduction(current_audio_path, temp_dir, config)
        if nr_path and os.path.exists(nr_path):
            print(f"{Colors.GREEN}✅ Noise Reduction applied: {os.path.basename(nr_path)}{Colors.RESET}")
            current_audio_path = nr_path
        else:
            print(f"{Colors.YELLOW}⚠️ Noise reduction failed or skipped. Proceeding without it.{Colors.RESET}")

    # 3. Apply Silero VAD (Optional)
    if apply_vad:
        print_status("Applying Silero VAD...")
        vad_path = apply_vad_silero(current_audio_path, temp_dir, config)
        if vad_path and os.path.exists(vad_path):
            print(f"{Colors.GREEN}✅ VAD applied: {os.path.basename(vad_path)}{Colors.RESET}")
            current_audio_path = vad_path
        else:
             print(f"{Colors.YELLOW}⚠️ VAD failed or skipped. Proceeding without it.{Colors.RESET}")

    # Return the path of the audio file after all successful processing steps
    if current_audio_path and os.path.exists(current_audio_path):
         return current_audio_path
    else:
         print_error("Final audio path is missing or invalid after processing steps.")
         logger.error("Final audio path is missing or invalid after processing steps.")
         return None


# --- Updated Main Workflow Function ---
def run_classification(source_type, source_data, apply_nr, apply_vad):
    """Main workflow: process audio source, classify, and display results with UX."""
    global MODEL_READY, global_classifier
    try:
         if not MODEL_READY or not global_classifier:
             print_error("Model is not ready. Please run Cell 1-3 successfully first.")
             logger.error("Attempted to run classification while model not ready.")
             return
    except NameError:
          print_error("Model readiness check failed (MODEL_READY or global_classifier not found). Please run Cells 1-3.")
          return

    print_header(f"Starting Classification: {source_type.capitalize()} Source")
    print(f"{Colors.BLUE}  ├─ Noise Reduction: {'Enabled' if apply_nr else 'Disabled'}{Colors.RESET}")
    print(f"{Colors.BLUE}  └─ Silero VAD:      {'Enabled' if apply_vad else 'Disabled'}{Colors.RESET}")
    # print("="*30) # Replaced by header

    result = {"error": "Initialization failed"} # Default error
    # Use the temporary directory context manager for automatic cleanup
    with temporary_directory(prefix="accent_clf_") as temp_dir:
        logger.info(f"Using temporary directory: {temp_dir}")

        # --- Step 1: Process Audio Source ---
        print_status("Processing audio source...")
        final_audio_path = _process_audio_source(
            source_type, source_data, temp_dir, CONFIG, apply_nr, apply_vad
        )

        # --- Step 2: Classify ---
        if final_audio_path: # Check if processing was successful
            print_status("Classifying accent (this may take a moment)...")
            # Pass the temp_dir for the optimizer (part of predict_accent) to use
            result = global_classifier.predict_accent(final_audio_path, temp_dir)
        else:
             logger.error("Audio processing pipeline failed before classification.")
             # Update result dict with specific error
             result = {"error": "Audio processing failed before classification step. Check logs above."}
             # Display error immediately since processing stopped
             _display_results(result)
             print(f"\n{Colors.YELLOW}Classification aborted due to processing error.{Colors.RESET}")
             return # Exit early

        # --- Step 3: Display ---
        _display_results(result)
        return(result)
    # Context manager handles cleanup of temp_dir automatically
    print(f"\n{Colors.GREEN}✅ Classification process complete.{Colors.RESET}")

print("✅ Core processing functions defined")

✅ Core processing functions defined


In [ ]:
#@title 5. Classify Accent from YouTube/Video URL
#@markdown Enter the URL of a video/audio (e.g., YouTube) containing English speech.

# --- Input ---
video_url = "https://vimeo.com/1060452212" #@param {type:"string"}

# --- Preprocessing Options ---
#@markdown Enable basic Noise Reduction (attempts to reduce background noise using torchaudio VAD):
use_noise_reduction = True #@param {type:"boolean"}
#@markdown Enable Silero VAD (Voice Activity Detection) to try removing silence:
use_vad = False #@param {type:"boolean"}
#@markdown ---

if not video_url:
    print("🤔 Please enter a valid video URL above and run this cell again.")
else:
    # Run the main classification workflow
    run_classification(
        source_type='url',
        source_data=video_url,
        apply_nr=use_noise_reduction,
        apply_vad=use_vad
    )


========== Starting Classification: Url Source ==========
  ├─ Noise Reduction: Enabled
  └─ Silero VAD:      Disabled
⏳ Processing audio source...
⏳ Downloading/Extracting audio from URL...


✅ Initial audio ready: audio_08ef3421-a589-4bef-8ed9-d9dd2ca6f173.wav
⏳ Applying Noise Reduction...
✅ Noise Reduction applied: nr_vad_audio_08ef3421-a589-4bef-8ed9-d9dd2ca6f173.wav
⏳ Classifying accent (this may take a moment)...


ERROR:__main__:Unexpected ffprobe output format: ['16000\n1\n14.824000']



========== Classification Results ==========
🗣️ Predicted Accent: Singaporean English
📊 Confidence: 61.4%

📝 Explanation:
The model predicts the speaker has a **Singaporean English** accent with moderate confidence (61.4%). The next likely accent detected is **Indian English** (27.0%).

🔝 Top 5 Predictions:
  1. Singaporean English (61.4%)
  2. Indian English (27.0%)
  3. American English (11.4%)
  4. African English (0.1%)
  5. South Atlantic English (0.0%)

✅ Classification process complete.


In [ ]:
#@title 6. Classify Accent from Uploaded Audio File
#@markdown Run this cell to upload an audio file (e.g., `.wav`, `.mp3`, `.m4a`).

# --- Preprocessing Options ---
#@markdown Enable basic Noise Reduction (attempts to reduce background noise using torchaudio VAD):
upload_use_noise_reduction = False #@param {type:"boolean"}
#@markdown Enable Silero VAD (Voice Activity Detection) to try removing silence:
upload_use_vad = False #@param {type:"boolean"}
#@markdown ---

print("Please upload an audio file...")
uploaded = files.upload()

if not uploaded:
  print("\\nNo file uploaded or upload canceled.")
else:
  # Get the first uploaded file's name and content
  file_name = list(uploaded.keys())[0]
  file_content = uploaded[file_name]
  print(f"\\nProcessing uploaded file: {file_name}")

  # Run the main classification workflow for the uploaded file
  run_classification(
      source_type='file',
      source_data=(file_name, file_content), # Pass filename and content
      apply_nr=upload_use_noise_reduction,
      apply_vad=upload_use_vad
  )

In [ ]:
#@title 7. Batch Classify Accents from Multiple URLs (File Upload)
#@markdown Upload a `.txt` file containing one video/audio URL per line.
#@markdown The script will process each URL sequentially.

from google.colab import files
import io # Needed for reading file content
import csv # Needed for writing results to CSV
import os # Needed for path joining
import datetime # Needed for timestamp in filename

# --- Configuration ---
#@markdown Define the base name for the output results file:
output_filename_base = "batch_results" #@param {type:"string"}

# --- Preprocessing Options ---
#@markdown Enable basic Noise Reduction (attempts to reduce background noise using torchaudio VAD):
use_noise_reduction = True #@param {type:"boolean"}
#@markdown Enable Silero VAD (Voice Activity Detection) to try removing silence:
use_vad = False #@param {type:"boolean"}
#@markdown ---

urls_to_process = []
uploaded_filename = None

print("Please upload a '.txt' file containing the list of URLs (one URL per line).")
uploaded = files.upload()

if not uploaded:
    print("🚫 No file uploaded. Please run the cell again and upload a file.")
else:
    # Process the first uploaded .txt file found
    for fn, content in uploaded.items():
        if fn.lower().endswith('.txt'):
            uploaded_filename = fn
            print(f"\nProcessing uploaded file: {uploaded_filename}")
            try:
                # Decode the byte content into a string (assuming UTF-8)
                text_content = content.decode('utf-8')
                # Split into lines, strip whitespace, and filter empty lines
                urls_to_process = [url.strip() for url in text_content.strip().splitlines() if url.strip()]
                break # Stop after processing the first .txt file
            except UnicodeDecodeError:
                print(f"❌ Error: Could not decode file '{uploaded_filename}' as UTF-8. Please ensure it's a plain text file.")
                urls_to_process = [] # Reset list on error
                break
            except Exception as e:
                 print(f"❌ Error reading file '{uploaded_filename}': {e}")
                 urls_to_process = [] # Reset list on error
                 break
        else:
             print(f"ℹ️ Skipping non-txt file: {fn}")

    if not uploaded_filename:
         print("🚫 No '.txt' file was found in the upload.")
    elif not urls_to_process and uploaded_filename:
         print(f"🤔 The file '{uploaded_filename}' seemed empty or contained no valid URLs after processing.")


# --- Run Classification if URLs were loaded ---
if not urls_to_process:
    print("\nNo URLs to process from the uploaded file.")
else:
    print(f"\n🚀 Starting batch processing for {len(urls_to_process)} URL(s) from '{uploaded_filename}'...")
    print("-" * 40)

    # Ensure the run_classification function is defined and available in your environment
    run_classification_defined = 'run_classification' in globals()
    if not run_classification_defined:
         print("❌ Error: The 'run_classification' function is not defined.")
         print("Please make sure you have run the cell that defines this function.")

    # Ensure logger is defined (add basic config if needed)
    logger_defined = 'logger' in globals()
    if not logger_defined:
         import logging
         logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
         logger = logging.getLogger(__name__)
         logger_defined = True # Mark as defined now
         print("ℹ️ Basic logger initialized.")

    # Proceed only if run_classification is defined
    if run_classification_defined:
        successful_count = 0
        error_count = 0

        # --- Prepare CSV file ---
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_csv_filename = f"{output_filename_base}_{timestamp}.csv"
        # Define the header based on expected keys from run_classification/predict_accent
        # Assuming run_classification returns a dict similar to predict_accent's output
        csv_fieldnames = ['URL', 'Accent', 'Confidence', 'Top_Accents', 'Explanation', 'Error']

        try:
            with open(output_csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=csv_fieldnames)
                writer.writeheader()
                print(f"📝 Saving results to: {output_csv_filename}")
                print("-" * 40)

                # --- Process each URL ---
                for i, url in enumerate(urls_to_process):
                    print(f"\nProcessing URL {i+1}/{len(urls_to_process)}: {url}")
                    result_data = {'URL': url} # Initialize row data with URL
                    try:
                        # --- IMPORTANT ASSUMPTION ---
                        # Assumes `run_classification` RETURNS a dictionary containing the results
                        # (e.g., {'accent': '...', 'confidence': ..., 'top_accents': [...], 'explanation': '...'})
                        # or an error dictionary (e.g., {'error': '...'}).
                        # If `run_classification` only prints, you need to modify it to return results.
                        result = run_classification(
                            source_type='url',
                            source_data=url,
                            apply_nr=use_noise_reduction,
                            apply_vad=use_vad
                        )

                        if result and 'error' in result:
                             # Handle errors reported by run_classification
                             error_count += 1
                             error_message = result.get('error', 'Unknown error from run_classification')
                             result_data['Error'] = error_message
                             print(f"⚠️ Error reported by classification function: {error_message}")
                             if logger_defined: logger.warning(f"Classification error for URL '{url}': {error_message}")

                        elif result:
                            # Process successful result
                            successful_count += 1
                            result_data['Accent'] = result.get('accent')
                            result_data['Confidence'] = result.get('confidence')
                            # Convert list of top accents to a string for CSV
                            result_data['Top_Accents'] = str(result.get('top_accents', []))
                            result_data['Explanation'] = result.get('explanation')
                            print(f"✅ Result: {result.get('accent')} ({result.get('confidence')}%)")
                        else:
                             # Handle case where run_classification returns None or empty
                             error_count += 1
                             error_message = "Classification function returned no result."
                             result_data['Error'] = error_message
                             print(f"⚠️ {error_message}")
                             if logger_defined: logger.warning(f"No result returned for URL '{url}'")

                    except Exception as e:
                        # Handle exceptions during the run_classification call itself
                        error_count += 1
                        error_message = f"Exception during processing: {str(e)}"
                        result_data['Error'] = error_message
                        print(f"❌ Exception processing URL: {url}. Error: {e}")
                        if logger_defined:
                            logger.error(f"Exception processing URL '{url}': {e}", exc_info=True)
                        else:
                            print(f"Logger not found. Exception processing URL '{url}': {e}")
                        print(f"❌ Error processing URL: {url}. See logs or error messages above. Skipping...") # Also print to cell output

                    # Write the row to the CSV file
                    writer.writerow(result_data)
                    print("-" * 40) # Separator between results

            print("\n✅ Batch processing complete.")
            print(f"📊 Summary: {successful_count} URLs processed successfully, {error_count} encountered errors.")
            print(f"💾 Results saved to: {output_csv_filename}")
            # Offer download link in Colab
            try:
                files.download(output_csv_filename)
            except NameError:
                print("\n(Could not automatically trigger download - 'files.download' not available outside Colab)")
            except Exception as download_err:
                print(f"\n(Could not automatically trigger download - Error: {download_err})")

        except IOError as io_err:
             print(f"\n❌ Error opening or writing to CSV file '{output_csv_filename}': {io_err}")
        except Exception as outer_err:
             print(f"\n❌ An unexpected error occurred during the batch process: {outer_err}")
             if logger_defined: logger.error("Unexpected error during batch process", exc_info=True)

Please upload a '.txt' file containing the list of URLs (one URL per line).


Saving test.txt to test (2).txt

Processing uploaded file: test (2).txt

🚀 Starting batch processing for 2 URL(s) from 'test (2).txt'...
----------------------------------------
📝 Saving results to: batch_results_20250502_204657.csv
----------------------------------------

Processing URL 1/2: https://www.youtube.com/watch?v=hIzjHBwjGCk&ab_channel=MarquesBrownlee

========== Starting Classification: Url Source ==========
  ├─ Noise Reduction: Enabled
  └─ Silero VAD:      Disabled
⏳ Processing audio source...
⏳ Downloading/Extracting audio from URL...


✅ Initial audio ready: audio_c4c3460b-f8df-448c-947f-27e663827409.wav
⏳ Applying Noise Reduction...
✅ Noise Reduction applied: nr_vad_audio_c4c3460b-f8df-448c-947f-27e663827409.wav
⏳ Classifying accent (this may take a moment)...


ERROR:__main__:Unexpected ffprobe output format: ['16000\n1\n630.313313']



========== Classification Results ==========
🗣️ Predicted Accent: American English
📊 Confidence: 100.0%

📝 Explanation:
The model predicts the speaker has a **American English** accent with very high confidence (100.0%). (Typically features rhoticity (pronounced 'r's) and distinct vowel sounds.)

🔝 Top 5 Predictions:
  1. American English (100.0%)
  2. Canadian English (0.0%)
  3. Australian English (0.0%)
  4. Irish English (0.0%)
  5. British English (0.0%)
✅ Result: American English (100.0%)
----------------------------------------

Processing URL 2/2: https://www.youtube.com/watch?v=LSoNSVKTvGw&ab_channel=RobertForbes

========== Starting Classification: Url Source ==========
  ├─ Noise Reduction: Enabled
  └─ Silero VAD:      Disabled
⏳ Processing audio source...
⏳ Downloading/Extracting audio from URL...


✅ Initial audio ready: audio_276ff8b6-020b-4a89-bf41-33aea7816fd0.wav
⏳ Applying Noise Reduction...
✅ Noise Reduction applied: nr_vad_audio_276ff8b6-020b-4a89-bf41-33aea7816fd0.wav
⏳ Classifying accent (this may take a moment)...


ERROR:__main__:Unexpected ffprobe output format: ['16000\n1\n51.404813']



========== Classification Results ==========
🗣️ Predicted Accent: American English
📊 Confidence: 99.6%

📝 Explanation:
The model predicts the speaker has a **American English** accent with very high confidence (99.6%). (Typically features rhoticity (pronounced 'r's) and distinct vowel sounds.)

🔝 Top 5 Predictions:
  1. American English (99.6%)
  2. Canadian English (0.4%)
  3. British English (0.0%)
  4. Scottish English (0.0%)
  5. Irish English (0.0%)
✅ Result: American English (99.6%)
----------------------------------------

✅ Batch processing complete.
📊 Summary: 2 URLs processed successfully, 0 encountered errors.
💾 Results saved to: batch_results_20250502_204657.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>